# Task_A _EXP_S_R09: Chain of thought with RAG Summarisation with openai/gpt-oss-120b

This notebook runs **Experiment: Chain of thought prompting summarization
with openai/gpt-oss-120b and with RAG**

In [1]:
# ================================================================
# INSTALL REQUIRED LIBRARIES
# ================================================================
!pip install -q pandas numpy rouge-score sentence-transformers scikit-learn groq tenacity tqdm chromadb

print("=" * 60)
print("Required Libraries installed")
print("=" * 60)


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.6 MB/s eta 0:00:00
ERROR: pip

In [2]:
# ================================================================
# 1. BASIC PYTHON LIBRARIES
# ================================================================
import os
import re
import json
import uuid
import shutil
import warnings
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ================================================================
# 2. NLP / EVALUATION LIBRARIES
# ================================================================
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# ================================================================
# 3. VECTOR DATABASE
# ================================================================
import chromadb


# ================================================================
# 4. LLM LIBRARIES
# ================================================================
from groq import Groq, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type


# ================================================================
# 5. GOOGLE COLAB / DRIVE
# ================================================================
from google.colab import drive, userdata


# ================================================================
# 6. VERSION CHECKS
# ================================================================
def get_version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return "installed"


print("=" * 60)
print("LIBRARY VERSION CHECK")
print("=" * 60)
print("pandas               :", pd.__version__)
print("numpy                :", np.__version__)
print("rouge-score          :", get_version("rouge-score"))
print("sentence-transformers:", get_version("sentence-transformers"))
print("scikit-learn         :", get_version("scikit-learn"))
print("groq                 :", get_version("groq"))
print("chromadb             :", get_version("chromadb"))
print("=" * 60)


# ================================================================
# 7. MOUNT GOOGLE DRIVE
# ================================================================
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Masters/Masters_Thesis_Project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)


# ================================================================
# 8. LOAD GROQ API KEY
# ================================================================
GROQ_API_KEY = userdata.get("GROQ_API_KEY_3")
assert GROQ_API_KEY, "GROQ_API_KEY is missing in Colab Secrets."

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("GROQ API key loaded from Colab secrets")


# ================================================================
# 9. EXPERIMENT CONFIGURATION
# ================================================================
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "reference_summaries"
REFERENCE_DATASET_PATH = PROCESSED_DATA_DIR / "djinni_with_reference_summaries.csv"

TEXT_COLUMN = "Long Description"
ID_COLUMN = "id"
POSITION_COLUMN = "Position"
REFERENCE_SUMMARY_COLUMN = "Reference_Summary"

EXPERIMENT_ID = "exp_s_R09_summarization_chain_of_thought_chromadb_openai-gpt-oss-120b"
PROMPT_TYPE = "chain_of_thought_with_rag"
MODEL_NAME = "openai/gpt-oss-120b"
REASONING_EFFORT = "low"
TEMPERATURE = 0.0
MAX_TOKENS = 700

CHUNK_SIZE_WORDS = 150
CHUNK_OVERLAP_WORDS = 30
TOP_K = 5

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# For testing, use 10.
# For full experiment, set RUN_SAMPLE_SIZE = None
RUN_SAMPLE_SIZE = None

OUTPUT_DIR = PROJECT_DIR / "outputs" / "experiments" / "summarization_with_RAG" / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHROMA_DIR = OUTPUT_DIR / "chromadb_store"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

BATCH_RESULTS_JSON = OUTPUT_DIR / f"{EXPERIMENT_ID}.json"
BATCH_RESULTS_CSV = OUTPUT_DIR / f"{EXPERIMENT_ID}.csv"
FAILED_RESULTS_CSV = OUTPUT_DIR / f"{EXPERIMENT_ID}_failed.csv"

FINAL_METRICS_CSV = OUTPUT_DIR / f"{EXPERIMENT_ID}_final_metrics.csv"
FINAL_METRICS_JSON = OUTPUT_DIR / f"{EXPERIMENT_ID}_final_metrics.json"

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)
print("Experiment ID       :", EXPERIMENT_ID)
print("Prompt type         :", PROMPT_TYPE)
print("Model               :", MODEL_NAME)
print("Reasoning effort    :", REASONING_EFFORT)
print("Temperature         :", TEMPERATURE)
print("Max tokens          :", MAX_TOKENS)
print("Chunking strategy   : Fixed-size chunking")
print("Chunk size          :", CHUNK_SIZE_WORDS, "words")
print("Chunk overlap       :", CHUNK_OVERLAP_WORDS, "words")
print("Embedding model     :", EMBEDDING_MODEL_NAME)
print("Vector database     : ChromaDB")
print("Retrieval method    : Dense vector retrieval")
print("Similarity metric   : Cosine similarity")
print("Top-K               :", TOP_K)
print("Run sample size     :", RUN_SAMPLE_SIZE)
print("Reference CSV       :", REFERENCE_DATASET_PATH)
print("Output dir          :", OUTPUT_DIR)
print("=" * 60)


# ================================================================
# 10. LOAD DATASET
# ================================================================
print("=" * 60)
print("Loading processed Djinni dataset with reference summaries...")
print("=" * 60)

df = pd.read_csv(REFERENCE_DATASET_PATH)

print("Dataset loaded successfully")
print("Shape   :", df.shape)
print("Columns :", list(df.columns))
print("=" * 60)

display(df.head(3))


# ================================================================
# 11. DATA CLEANING / PREPROCESSING
# ================================================================
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def preprocess_dataset(df: pd.DataFrame, text_column: str, id_column: str) -> pd.DataFrame:
    work_df = df.copy()

    work_df = work_df[work_df[text_column].notna()]
    work_df = work_df[work_df[id_column].notna()]

    work_df[text_column] = work_df[text_column].apply(clean_text)

    work_df = work_df[work_df[text_column].str.strip().ne("")]
    work_df = work_df.drop_duplicates(subset=[text_column]).reset_index(drop=True)

    return work_df


print("Processing dataset...")

clean_df = preprocess_dataset(
    df=df,
    text_column=TEXT_COLUMN,
    id_column=ID_COLUMN
)

print("Original shape :", df.shape)
print("Cleaned shape  :", clean_df.shape)
print("Duplicates removed: Yes")
print("=" * 60)

display(clean_df[[ID_COLUMN, POSITION_COLUMN, TEXT_COLUMN]].head(3))


# ================================================================
# 12. FILTER ROWS WITH REFERENCE SUMMARIES
# ================================================================
if REFERENCE_SUMMARY_COLUMN not in clean_df.columns:
    raise ValueError(f"{REFERENCE_SUMMARY_COLUMN} column not found in dataset.")

clean_df[REFERENCE_SUMMARY_COLUMN] = clean_df[REFERENCE_SUMMARY_COLUMN].fillna("").astype(str)

clean_df.loc[
    clean_df[REFERENCE_SUMMARY_COLUMN].str.strip().str.lower() == "nan",
    REFERENCE_SUMMARY_COLUMN
] = ""

valid_df = clean_df[
    clean_df[REFERENCE_SUMMARY_COLUMN].str.strip().ne("")
].copy()

valid_df = valid_df.reset_index(drop=True)

if RUN_SAMPLE_SIZE is not None:
    experiment_df = valid_df.head(RUN_SAMPLE_SIZE).copy().reset_index(drop=True)
else:
    experiment_df = valid_df.copy().reset_index(drop=True)

print("Rows with reference summaries:", len(valid_df))
print("Experiment rows selected     :", len(experiment_df))
print("=" * 60)

display(experiment_df[[ID_COLUMN, POSITION_COLUMN]].head(10))


# ================================================================
# 13. FIXED-SIZE CHUNKING
# ================================================================
def split_into_fixed_word_chunks(
    text: str,
    chunk_size: int = 400,
    overlap: int = 50
):
    words = str(text).split()

    if len(words) == 0:
        return []

    chunks = []
    start = 0
    chunk_index = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)

        chunks.append({
            "chunk_index": chunk_index,
            "content": chunk_text,
            "start_word": start,
            "end_word": min(end, len(words))
        })

        chunk_index += 1

        if end >= len(words):
            break

        start = end - overlap

    return chunks


sample_chunks = split_into_fixed_word_chunks(
    experiment_df.loc[0, TEXT_COLUMN],
    chunk_size=CHUNK_SIZE_WORDS,
    overlap=CHUNK_OVERLAP_WORDS
)

print("Chunking test complete")
print("Sample JD ID       :", experiment_df.loc[0, ID_COLUMN])
print("Total chunks       :", len(sample_chunks))
print("Sample chunk words :", len(sample_chunks[0]["content"].split()) if sample_chunks else 0)
print("=" * 60)

# ================================================================
# 13A. CHUNK DISTRIBUTION CHECK
# ================================================================
chunk_counts = []
word_counts = []

for _, row in valid_df.iterrows():
    text = row[TEXT_COLUMN]
    words = str(text).split()
    chunks = split_into_fixed_word_chunks(
        text=text,
        chunk_size=CHUNK_SIZE_WORDS,
        overlap=CHUNK_OVERLAP_WORDS
    )

    word_counts.append(len(words))
    chunk_counts.append(len(chunks))

print("=" * 60)
print("CHUNK DISTRIBUTION CHECK")
print("=" * 60)
print("Total records        :", len(valid_df))
print("Chunk size           :", CHUNK_SIZE_WORDS)
print("Chunk overlap        :", CHUNK_OVERLAP_WORDS)
print("Avg words per JD     :", round(np.mean(word_counts), 2))
print("Min words per JD     :", int(np.min(word_counts)))
print("Max words per JD     :", int(np.max(word_counts)))
print("Avg chunks per JD    :", round(np.mean(chunk_counts), 2))
print("Min chunks per JD    :", int(np.min(chunk_counts)))
print("Max chunks per JD    :", int(np.max(chunk_counts)))
print("JDs with 1 chunk     :", sum(c == 1 for c in chunk_counts))
print("JDs with 2+ chunks   :", sum(c >= 2 for c in chunk_counts))
print("JDs with >3 chunks   :", sum(c > 3 for c in chunk_counts))
print("=" * 60)


# ================================================================
# 14. EMBEDDING MODEL
# ================================================================
print("Initialising embedding model...")
print("Model :", EMBEDDING_MODEL_NAME)

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

embedding_dimension = embedding_model.get_sentence_embedding_dimension()

print("Embedding model ready")
print("Dimension:", embedding_dimension)
print("=" * 60)


def embed_texts(texts):
    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embeddings


# ================================================================
# 15. CHROMADB HELPERS
# ================================================================
def build_chromadb_for_single_jd(
    selected_id: str,
    job_description: str,
    chroma_base_dir: Path
):
    chunks = split_into_fixed_word_chunks(
        text=job_description,
        chunk_size=CHUNK_SIZE_WORDS,
        overlap=CHUNK_OVERLAP_WORDS
    )

    if not chunks:
        raise ValueError("No chunks created for this job description.")

    chunk_texts = [chunk["content"] for chunk in chunks]
    chunk_embeddings = embed_texts(chunk_texts)

    collection_name = f"jd_{str(selected_id).replace('-', '_')}_{uuid.uuid4().hex[:8]}"
    collection_name = re.sub(r"[^a-zA-Z0-9_]", "_", collection_name)[:60]

    client = chromadb.PersistentClient(path=str(chroma_base_dir))

    collection = client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    ids = []
    metadatas = []

    for chunk in chunks:
        chunk_id = f"{selected_id}_chunk_{chunk['chunk_index']}"
        ids.append(chunk_id)
        metadatas.append({
            "selected_id": str(selected_id),
            "chunk_index": chunk["chunk_index"],
            "start_word": chunk["start_word"],
            "end_word": chunk["end_word"]
        })

    collection.add(
        ids=ids,
        documents=chunk_texts,
        embeddings=chunk_embeddings.tolist(),
        metadatas=metadatas
    )

    return client, collection, chunks, collection_name


def retrieve_top_k_chunks(collection, query_text: str, top_k: int = 5):
    query_embedding = embed_texts([query_text])[0]

    result = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]

    for rank, (doc, metadata, distance) in enumerate(zip(documents, metadatas, distances), start=1):
        similarity = 1 - float(distance)

        retrieved_chunks.append({
            "rank": rank,
            "chunk_id": f"{metadata['selected_id']}_chunk_{metadata['chunk_index']}",
            "chunk_index": metadata["chunk_index"],
            "content": doc,
            "distance": round(float(distance), 4),
            "similarity": round(similarity, 4)
        })

    return retrieved_chunks


def merge_retrieved_chunks(retrieved_chunks):
    context_parts = []

    for chunk in retrieved_chunks:
        context_parts.append(
            f"[Chunk {chunk['rank']} | Similarity: {chunk['similarity']}]\n{chunk['content']}"
        )

    return "\n\n".join(context_parts)


print("ChromaDB helper functions ready")
print("=" * 60)


# ================================================================
# 16. CHAIN OF THOUGHT RAG PROMPT
# ================================================================
def build_chain_of_thought_rag_prompt(
    retrieved_context: str,
    job_description: str
) -> str:
    return f"""
You are a professional HR analyst.

Your task is to create a structured HR summary using only the retrieved context and target job description.

Before writing the final answer, internally check:
1. What role is being described?
2. What responsibilities are explicitly stated?
3. What skills are explicitly mentioned?
4. What tools, technologies, frameworks, platforms, or databases are explicitly mentioned?
5. Whether experience is explicitly stated.
6. Whether every included item is grounded in the retrieved context or target job description.

Do NOT show your reasoning.
Do NOT explain your steps.
Output only the final structured summary.

Strict grounding rules:
- Use retrieved context as the primary evidence.
- Use target job description only to resolve missing or incomplete details.
- Do NOT use external knowledge.
- Do NOT infer missing skills.
- Do NOT add common industry skills unless explicitly mentioned.
- Do NOT paraphrase skills or tools unnecessarily.
- If information is missing, write exactly: Not explicitly stated.

Output format:

Role Overview:
<1 sentence describing the role>

Responsibilities:
<max 2 short sentences>

Skills:
<comma-separated list of explicitly mentioned skills>

Tools:
<comma-separated list of explicitly mentioned tools, technologies, frameworks, programming languages, databases, and platforms>

Experience:
<1 sentence with explicitly stated experience or "Not explicitly stated">

Retrieved Context:
{retrieved_context}

Target Job Description:
{job_description}

Final Answer:
""".strip()


# ================================================================
# 17. GROQ CLIENT
# ================================================================
class LLMClientError(Exception):
    pass


class GroqClient:
    def __init__(self, model_name: str):
        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY is missing from environment variables.")

        self.client = Groq(api_key=api_key)
        self.model_name = model_name

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type(Exception),
        reraise=True
    )
    def generate(
        self,
        prompt: str,
        temperature: float,
        max_tokens: int
    ) -> str:
        completion = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "You are a precise and factual HR summarisation assistant."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=temperature,
            max_tokens=max_tokens,
            reasoning_effort=REASONING_EFFORT
        )

        if not completion.choices:
            raise LLMClientError("Groq returned no choices.")

        text = completion.choices[0].message.content

        if not text or not str(text).strip():
            raise LLMClientError("Groq returned an empty response.")

        return str(text).strip()


print("Initialising LLM via Groq...")
groq_client = GroqClient(model_name=MODEL_NAME)
print("LLM ready:", MODEL_NAME)
print("=" * 60)


# ================================================================
# 18. CORE EVALUATION FUNCTIONS
# ================================================================
def compute_rouge_l(reference_text: str, generated_text: str):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    score = scorer.score(reference_text, generated_text)["rougeL"]

    return {
        "rougeL_precision": round(score.precision, 4),
        "rougeL_recall": round(score.recall, 4),
        "rougeL_f1": round(score.fmeasure, 4),
    }


def compute_semantic_similarity(text_1: str, text_2: str):
    embeddings = embed_texts([text_1, text_2])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return round(float(similarity), 4)


def compute_compression_ratio(original_text: str, summary_text: str):
    original_length = max(len(str(original_text).split()), 1)
    summary_length = len(str(summary_text).split())
    return round(summary_length / original_length, 4)


def validate_experiment_summary_sections(summary_text: str):
    required_sections = [
        "Role Overview",
        "Responsibilities",
        "Skills",
        "Tools",
        "Experience",
    ]

    lowered = str(summary_text).lower()

    return {
        section: f"{section.lower()}:" in lowered
        for section in required_sections
    }


print("Core evaluation functions ready")
print("=" * 60)


# ================================================================
# 19. SKILL EXTRACTION AND CUSTOM METRICS
# ================================================================
def normalize_skill(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^[,;:\-\.\s]+|[,;:\-\.\s]+$", "", text)
    return text


def extract_skills(text: str):
    if not text or not isinstance(text, str):
        return []

    text = re.sub(r"\s+", " ", text).strip()
    extracted = []

    patterns = [
        r"(?:experience with|experience in|proficient in|proficiency in|knowledge of|familiarity with|expertise in|skills in|skilled in)\s+([A-Za-z0-9\+\#\./,\-\s]+?)(?:\.|;|:|, and| and |\n|$)",
        r"(?:tools?:|technologies:|skills:)\s*([A-Za-z0-9\+\#\./,\-\s]+?)(?:\.|\n|$)",
        r"(?:requirements?:|required skills:)\s*([A-Za-z0-9\+\#\./,\-\s]+?)(?:\.|\n|$)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        for match in matches:
            parts = re.split(r",| and |/|\|", match)

            for part in parts:
                skill = normalize_skill(part)

                if skill and len(skill) > 1:
                    extracted.append(skill)

    tech_terms = re.findall(
        r"\b(?:[A-Z][A-Za-z0-9\+\#\.]{1,}|[A-Za-z]+\+[A-Za-z]+|[A-Za-z]+#[A-Za-z]+)\b",
        text
    )

    for term in tech_terms:
        skill = normalize_skill(term)

        if skill and len(skill) > 1:
            extracted.append(skill)

    blacklist = {
        "experience",
        "skills",
        "tools",
        "technologies",
        "knowledge",
        "familiarity",
        "expertise",
        "requirements",
        "required skills",
        "role overview",
        "responsibilities",
    }

    extracted = [s for s in extracted if s not in blacklist]

    return sorted(list(set(extracted)))


def build_skill_analysis(original_text: str, summary_text: str):
    jd_skills = extract_skills(original_text)
    summary_skills = extract_skills(summary_text)

    jd_set = set(map(normalize_skill, jd_skills))
    summary_set = set(map(normalize_skill, summary_skills))

    irr = len(jd_set & summary_set) / len(jd_set) if len(jd_set) > 0 else 0.0
    hallucinated_skills = sorted(list(summary_set - jd_set))

    return {
        "jd_skills": sorted(list(jd_set)),
        "summary_skills": sorted(list(summary_set)),
        "irr": round(irr, 4),
        "hallucinated_skills": hallucinated_skills,
    }


def compute_hallucination_rate(original_skills, summary_skills):
    original_set = set(map(normalize_skill, original_skills))
    summary_set = set(map(normalize_skill, summary_skills))

    if len(summary_set) == 0:
        return 0.0

    hallucinated = summary_set - original_set

    return round(len(hallucinated) / len(summary_set), 4)


def compute_top_k_skill_coverage(original_skills, summary_skills, k=5):
    original_top_k = list(original_skills)[:k]

    original_set = set(map(normalize_skill, original_top_k))
    summary_set = set(map(normalize_skill, summary_skills))

    if len(original_set) == 0:
        return 0.0

    return round(len(original_set & summary_set) / len(original_set), 4)


print("Skill extraction functions ready")
print("=" * 60)


# ================================================================
# 20. RAG-SPECIFIC METRICS
# ================================================================
def compute_grounding_score(generated_summary: str, retrieved_context: str):
    """
    Grounding score proxy:
    Semantic similarity between generated summary and retrieved chunks.
    Higher score means the summary is more aligned with retrieved context.
    """
    if not generated_summary or not retrieved_context:
        return 0.0

    return compute_semantic_similarity(generated_summary, retrieved_context)


def compute_faithfulness_score(generated_summary: str, original_jd: str):
    """
    Faithfulness score proxy:
    Semantic similarity between generated summary and original JD.
    Higher score means the summary is more faithful to the source JD.
    """
    if not generated_summary or not original_jd:
        return 0.0

    return compute_semantic_similarity(generated_summary, original_jd)


def compute_retrieval_similarity_stats(retrieved_chunks):
    similarities = [chunk["similarity"] for chunk in retrieved_chunks]

    if not similarities:
        return {
            "retrieval_similarity_mean": 0.0,
            "retrieval_similarity_max": 0.0,
            "retrieval_similarity_min": 0.0,
        }

    return {
        "retrieval_similarity_mean": round(float(np.mean(similarities)), 4),
        "retrieval_similarity_max": round(float(np.max(similarities)), 4),
        "retrieval_similarity_min": round(float(np.min(similarities)), 4),
    }


print("RAG-specific metric functions ready")
print("=" * 60)


# ================================================================
# 21. CHECKPOINT / RESUME
# ================================================================
if BATCH_RESULTS_JSON.exists():
    with open(BATCH_RESULTS_JSON, "r", encoding="utf-8") as f:
        loaded = json.load(f)

    if isinstance(loaded, list):
        all_results = loaded
        completed_ids = {str(r["selected_id"]) for r in all_results if "selected_id" in r}
    else:
        all_results = []
        completed_ids = set()
else:
    all_results = []
    completed_ids = set()


failed_results = []

if FAILED_RESULTS_CSV.exists():
    try:
        failed_results = pd.read_csv(FAILED_RESULTS_CSV).to_dict(orient="records")
    except Exception:
        failed_results = []


print("Already completed:", len(completed_ids))
print("Already failed   :", len(failed_results))
print("=" * 60)


# ================================================================
# 22. CHECKPOINT SAVE FUNCTION
# ================================================================
def save_checkpoints(all_results, failed_results):
    with open(BATCH_RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    if all_results:
        checkpoint_df = pd.DataFrame(all_results).copy()

        json_columns = [
            "jd_skills",
            "summary_skills",
            "hallucinated_skills",
            "section_validation",
            "retrieved_chunks",
        ]

        for col in json_columns:
            if col in checkpoint_df.columns:
                checkpoint_df[col] = checkpoint_df[col].apply(
                    lambda x: json.dumps(x, ensure_ascii=False)
                )

        checkpoint_df.to_csv(BATCH_RESULTS_CSV, index=False, encoding="utf-8")

    if failed_results:
        pd.DataFrame(failed_results).to_csv(
            FAILED_RESULTS_CSV,
            index=False,
            encoding="utf-8"
        )


print("Checkpoint save function ready")
print("=" * 60)


# ================================================================
# 23. RUN CHAIN OF THOUGHT RAG EXPERIMENT
# ================================================================
print("=" * 60)
print("Running Chain of thought RAG Experiment")
print("=" * 60)
print("Samples :", len(experiment_df))
print("Top-K   :", TOP_K)
print("Model   :", MODEL_NAME)
print("=" * 60)


for i, row in tqdm(
    experiment_df.iterrows(),
    total=len(experiment_df),
    desc="Running Chain of thought RAG Experiment",
    unit="JD"
):
    current_id = str(row[ID_COLUMN])

    if current_id in completed_ids:
        tqdm.write(f"Skipped {i+1}/{len(experiment_df)} | ID: {current_id}")
        continue

    tqdm.write("=" * 60)
    tqdm.write(f"Running {i+1}/{len(experiment_df)} | ID: {current_id}")

    try:
        job_description = row[TEXT_COLUMN]
        reference_summary = row[REFERENCE_SUMMARY_COLUMN]
        position = row[POSITION_COLUMN] if POSITION_COLUMN in row else ""

        # 1. Build ChromaDB for selected JD
        client, collection, chunks, collection_name = build_chromadb_for_single_jd(
            selected_id=current_id,
            job_description=job_description,
            chroma_base_dir=CHROMA_DIR
        )

        tqdm.write(f"Chunking complete | Total chunks: {len(chunks)}")
        tqdm.write(f"ChromaDB collection created: {collection_name}")

        # 2. Form retrieval query from target JD
        retrieval_query = job_description

        # 3. Retrieve Top-K chunks
        retrieved_chunks = retrieve_top_k_chunks(
            collection=collection,
            query_text=retrieval_query,
            top_k=TOP_K
        )

        tqdm.write(f"Retrieved {len(retrieved_chunks)} chunks")

        for chunk in retrieved_chunks[:3]:
            preview = chunk["content"][:100].replace("\n", " ")
            tqdm.write(
                f"Rank {chunk['rank']} | "
                f"score={chunk['similarity']} | "
                f"{preview}..."
            )

        retrieved_context = merge_retrieved_chunks(retrieved_chunks)

        # 4. Build a Chain of thought RAG prompt
        prompt = build_chain_of_thought_rag_prompt(
            retrieved_context=retrieved_context,
            job_description=job_description
        )

        # 5. Generate summary
        generated_summary = groq_client.generate(
            prompt=prompt,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        # 6. Core metrics
        rouge_scores = compute_rouge_l(
            reference_text=reference_summary,
            generated_text=generated_summary
        )

        semantic_similarity = compute_semantic_similarity(
            text_1=reference_summary,
            text_2=generated_summary
        )

        compression_ratio = compute_compression_ratio(
            original_text=job_description,
            summary_text=generated_summary
        )

        # 7. Skill metrics
        skill_analysis = build_skill_analysis(
            original_text=job_description,
            summary_text=generated_summary
        )

        hallucination_rate = compute_hallucination_rate(
            original_skills=skill_analysis["jd_skills"],
            summary_skills=skill_analysis["summary_skills"]
        )

        top_k_skill_coverage = compute_top_k_skill_coverage(
            original_skills=skill_analysis["jd_skills"],
            summary_skills=skill_analysis["summary_skills"],
            k=5
        )

        # 8. RAG-specific metrics
        grounding_score = compute_grounding_score(
            generated_summary=generated_summary,
            retrieved_context=retrieved_context
        )

        faithfulness_score = compute_faithfulness_score(
            generated_summary=generated_summary,
            original_jd=job_description
        )

        retrieval_stats = compute_retrieval_similarity_stats(retrieved_chunks)

        section_validation = validate_experiment_summary_sections(
            generated_summary
        )

        result = {
            "experiment_id": EXPERIMENT_ID,
            "selected_id": current_id,
            "position": position,
            "prompt_type": PROMPT_TYPE,
            "model_name": MODEL_NAME,
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS,

            "chunking_strategy": "fixed_size_word_chunking",
            "chunk_size_words": CHUNK_SIZE_WORDS,
            "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
            "embedding_model": EMBEDDING_MODEL_NAME,
            "vector_database": "ChromaDB",
            "retrieval_method": "dense_vector_retrieval",
            "similarity_metric": "cosine",
            "top_k": TOP_K,
            "num_chunks_created": len(chunks),

            "job_description": job_description,
            "retrieved_context": retrieved_context,
            "retrieved_chunks": retrieved_chunks,
            "reference_summary": reference_summary,
            "generated_summary": generated_summary,

            **rouge_scores,
            "semantic_similarity": semantic_similarity,
            "compression_ratio": compression_ratio,

            "jd_skills": skill_analysis["jd_skills"],
            "summary_skills": skill_analysis["summary_skills"],
            "irr": skill_analysis["irr"],
            "hallucinated_skills": skill_analysis["hallucinated_skills"],
            "hallucination_rate": hallucination_rate,
            "top_k_skill_coverage": top_k_skill_coverage,

            "grounding_score": grounding_score,
            "faithfulness_score": faithfulness_score,
            **retrieval_stats,

            "section_validation": section_validation,
        }

        all_results.append(result)
        completed_ids.add(current_id)

        save_checkpoints(all_results, failed_results)

        tqdm.write(
            f"Completed {i+1}/{len(experiment_df)} | "
            f"ID: {current_id} | "
            f"ROUGE-L: {result['rougeL_f1']} | "
            f"Sim: {result['semantic_similarity']} | "
            f"IRR: {result['irr']} | "
            f"Grounding: {result['grounding_score']} | "
            f"Faithfulness: {result['faithfulness_score']}"
        )

    except RateLimitError as e:
        tqdm.write("Rate limit hit. Checkpoint saved.")
        tqdm.write(str(e))
        save_checkpoints(all_results, failed_results)
        break

    except Exception as e:
        tqdm.write(
            f"Failed {i+1}/{len(experiment_df)} | "
            f"ID: {current_id} | Error: {e}"
        )

        failed_results.append({
            "selected_id": current_id,
            "error": str(e)
        })

        save_checkpoints(all_results, failed_results)
        continue


print("=" * 60)
print("Pipeline complete")
print("Successful results:", len(all_results))
print("Failed results    :", len(failed_results))
print("=" * 60)


# ================================================================
# 24. FINAL SUMMARY METRICS
# ================================================================
results_df = pd.DataFrame(all_results)

print("Saved JSON to:", BATCH_RESULTS_JSON)
print("Saved CSV to :", BATCH_RESULTS_CSV)

METRIC_COLUMNS = [
    "rougeL_f1",
    "semantic_similarity",
    "irr",
    "hallucination_rate",
    "compression_ratio",
    "grounding_score",
    "faithfulness_score",
    "top_k_skill_coverage",
    "retrieval_similarity_mean",
]


def compute_global_skill_overlap(results_df):
    all_jd_skills = set()
    all_summary_skills = set()

    for _, row in results_df.iterrows():
        jd_skills = row["jd_skills"]
        summary_skills = row["summary_skills"]

        if isinstance(jd_skills, list):
            all_jd_skills.update(jd_skills)

        if isinstance(summary_skills, list):
            all_summary_skills.update(summary_skills)

    if len(all_jd_skills) == 0:
        return 0.0

    overlap = all_jd_skills & all_summary_skills

    return round(len(overlap) / len(all_jd_skills), 4)


def compute_hallucination_prevalence(results_df):
    hallucinated_count = results_df["hallucinated_skills"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    summaries_with_hallucination = (hallucinated_count > 0).sum()
    total_summaries = len(results_df)

    prevalence = summaries_with_hallucination / total_summaries if total_summaries > 0 else 0.0

    affected = hallucinated_count[hallucinated_count > 0]
    density = affected.mean() if len(affected) > 0 else 0.0

    return round(prevalence, 4), round(density, 4)


if not results_df.empty:
    final_metrics = {
        "experiment_id": EXPERIMENT_ID,
        "prompt_type": PROMPT_TYPE,
        "model_name": MODEL_NAME,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "chunk_size_words": CHUNK_SIZE_WORDS,
        "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "vector_database": "ChromaDB",
        "retrieval_method": "dense_vector_retrieval",
        "similarity_metric": "cosine",
        "top_k": TOP_K,
        "num_samples_completed": len(results_df),
    }

    for metric in METRIC_COLUMNS:
        final_metrics[f"{metric}_mean"] = round(results_df[metric].mean(), 4)
        final_metrics[f"{metric}_median"] = round(results_df[metric].median(), 4)
        final_metrics[f"{metric}_std"] = round(results_df[metric].std(), 4)
        final_metrics[f"{metric}_min"] = round(results_df[metric].min(), 4)
        final_metrics[f"{metric}_max"] = round(results_df[metric].max(), 4)

    global_skill_overlap = compute_global_skill_overlap(results_df)
    hallucination_prevalence, hallucination_density = compute_hallucination_prevalence(results_df)

    final_metrics["global_skill_overlap"] = global_skill_overlap
    final_metrics["hallucination_prevalence"] = hallucination_prevalence
    final_metrics["hallucination_density"] = hallucination_density

    final_metrics_df = pd.DataFrame([final_metrics])

    final_metrics_df.to_csv(FINAL_METRICS_CSV, index=False, encoding="utf-8")

    with open(FINAL_METRICS_JSON, "w", encoding="utf-8") as f:
        json.dump(final_metrics, f, indent=4, ensure_ascii=False)

    print("\n" + "=" * 60)
    print("FINAL SUMMARY METRICS — CHAIN OF THOUGHT RAG — CHROMADB")
    print("=" * 60)
    print("ROUGE-L Mean              :", final_metrics["rougeL_f1_mean"])
    print("ROUGE-L Median            :", final_metrics["rougeL_f1_median"])
    print("ROUGE-L Std               :", final_metrics["rougeL_f1_std"])
    print("Semantic Similarity Mean  :", final_metrics["semantic_similarity_mean"])
    print("IRR Mean                  :", final_metrics["irr_mean"])
    print("Hallucination Rate Mean   :", final_metrics["hallucination_rate_mean"])
    print("Compression Ratio Mean    :", final_metrics["compression_ratio_mean"])
    print("Grounding Score Mean      :", final_metrics["grounding_score_mean"])
    print("Faithfulness Score Mean   :", final_metrics["faithfulness_score_mean"])
    print("Top-K Skill Coverage Mean :", final_metrics["top_k_skill_coverage_mean"])
    print("Global Skill Overlap      :", final_metrics["global_skill_overlap"])
    print("Hallucination Prevalence  :", final_metrics["hallucination_prevalence"])
    print("Hallucination Density     :", final_metrics["hallucination_density"])
    print("=" * 60)

    print("Saved final metrics CSV :", FINAL_METRICS_CSV)
    print("Saved final metrics JSON:", FINAL_METRICS_JSON)

    display(final_metrics_df)

else:
    print("No results available yet.")

LIBRARY VERSION CHECK
pandas               : 2.2.2
numpy                : 2.0.2
rouge-score          : 0.1.2
sentence-transformers: 5.4.1
scikit-learn         : 1.6.1
groq                 : 1.2.0
chromadb             : 1.5.8
Mounted at /content/drive
Project directory: /content/drive/MyDrive/Colab Notebooks/Masters/Masters_Thesis_Project
GROQ API key loaded from Colab secrets
EXPERIMENT CONFIGURATION
Experiment ID       : exp_s_R09_summarization_chain_of_thought_chromadb_openai-gpt-oss-120b
Prompt type         : chain_of_thought_with_rag
Model               : openai/gpt-oss-120b
Reasoning effort    : low
Temperature         : 0.0
Max tokens          : 700
Chunking strategy   : Fixed-size chunking
Chunk size          : 150 words
Chunk overlap       : 30 words
Embedding model     : sentence-transformers/all-MiniLM-L6-v2
Vector database     : ChromaDB
Retrieval method    : Dense vector retrieval
Similarity metric   : Cosine similarity
Top-K               : 5
Run sample size     : None
Ref

,Position,Long Description,Company Name,Exp Years,Primary Keyword,English Level,Published,Long Description_lang,id,index_level_0,Reference_Summary
0,Java Developer,The job holder will . Design develop and maint...,Quickstarter,3y,Java,upper,2021-12-01T00:00:00+02:00,en,67aefec1-1efc-5c5b-b2db-c0ef5b74f519,77261,Role Overview:\nThe Java Developer will design...
1,Integration specialist / DevOps,Project Description Join our Development Cente...,Luxoft,3y,DevOps,upper,2022-04-01T00:00:00+03:00,en,2ec599d6-1509-5c34-9aec-54ac2e04b2d5,72279,Role Overview:\nThe Integration Specialist / D...
2,Head of Marketing,TurnKey Labs is the most trusted offshore soft...,TurnKey Labs,5y,Marketing,upper,2021-10-01T00:00:00+03:00,en,edba8bfd-8a91-5b37-9f72-bde92397e804,69706,"Role Overview:\nReporting to the CEO, the Head..."


Processing dataset...
Original shape : (100, 11)
Cleaned shape  : (100, 11)
Duplicates removed: Yes


,id,Position,Long Description
0,67aefec1-1efc-5c5b-b2db-c0ef5b74f519,Java Developer,The job holder will . Design develop and maint...
1,2ec599d6-1509-5c34-9aec-54ac2e04b2d5,Integration specialist / DevOps,Project Description Join our Development Cente...
2,edba8bfd-8a91-5b37-9f72-bde92397e804,Head of Marketing,TurnKey Labs is the most trusted offshore soft...


Rows with reference summaries: 100
Experiment rows selected     : 100


...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready
Dimension: 384
ChromaDB helper functions ready
Initialising LLM via Groq...
LLM ready: openai/gpt-oss-120b
Core evaluation functions ready
Skill extraction functions ready
RAG-specific metric functions ready
Already completed: 0
Already failed   : 0
Checkpoint save function ready
Running Chain of thought RAG Experiment
Samples : 100
Top-K   : 5
Model   : openai/gpt-oss-120b


Running Chain of thought RAG Experiment:   0%|          | 0/100 [00:00<?, ?JD/s]

Running 1/100 | ID: 67aefec1-1efc-5c5b-b2db-c0ef5b74f519
Chunking complete | Total chunks: 1
ChromaDB collection created: jd_67aefec1_1efc_5c5b_b2db_c0ef5b74f519_70c7688d
Retrieved 1 chunks
Rank 1 | score=1.0 | The job holder will . Design develop and maintain back-end of large web-based applications with Java...
Completed 1/100 | ID: 67aefec1-1efc-5c5b-b2db-c0ef5b74f519 | ROUGE-L: 0.44 | Sim: 0.8518 | IRR: 0.3929 | Grounding: 0.6781 | Faithfulness: 0.6846
Running 2/100 | ID: 2ec599d6-1509-5c34-9aec-54ac2e04b2d5
Chunking complete | Total chunks: 2
ChromaDB collection created: jd_2ec599d6_1509_5c34_9aec_54ac2e04b2d5_ddf6d04a
Retrieved 2 chunks
Rank 1 | score=0.8755 | Project Description Join our Development Center and become a member of our open minded progressive a...
Rank 2 | score=0.7959 | plan maintain and communicate regular maintenance activities with key business and internal stakehol...
Completed 2/100 | ID: 2ec599d6-1509-5c34-9aec-54ac2e04b2d5 | ROUGE-L: 0.5136 | Sim: 0.8878 | 

,experiment_id,prompt_type,model_name,temperature,max_tokens,chunk_size_words,chunk_overlap_words,embedding_model,vector_database,retrieval_method,...,top_k_skill_coverage_min,top_k_skill_coverage_max,retrieval_similarity_mean_mean,retrieval_similarity_mean_median,retrieval_similarity_mean_std,retrieval_similarity_mean_min,retrieval_similarity_mean_max,global_skill_overlap,hallucination_prevalence,hallucination_density
0,exp_s_R09_summarization_chain_of_thought_chrom...,chain_of_thought_with_rag,openai/gpt-oss-120b,0.0,700,150,30,sentence-transformers/all-MiniLM-L6-v2,ChromaDB,dense_vector_retrieval,...,0.0,1.0,0.8168,0.8181,0.143,0.5401,1.0,0.4037,1.0,6.07
